# Unit 2 Assessment

In this assignment, we will focus on salary prediction. The data set for this assignment includes information on job postings. Use this data set to see if you can predict the salary of a job posting (i.e., the `Salary` column in the data set) based on the job description and other attributes of the job. This is important, because this model can make a salary recommendation as soon as a job posting is entered into a system.

## Description of Variables

The description of variables are provided in "Jobs - Data Dictionary.docx"

## Goal

Use the **jobs_alldata.csv** data set and build models to predict **salary**.

**Be careful: this is a REGRESSION task**

## Submission:

Please save and submit this Jupyter notebook file. The correctness of the code matters for your grade. **Readability and organization of your code is also important.** You may lose points for submitting unreadable/undecipherable code. Therefore, use markdown cells to create sections, and use comments where necessary.


## Important hints:

* This assignment requires you to work with a text-based column in addition to regular numeric/categorical columns. So you will have to pay attention to a few things during data processing.
* If you open the data in Excel to get familiar with it, don't save it before closing (otherwise, it messes up the values of one of the columns).
* You can do your data prep before or after the train/test split. Regardless, you should use train_test_split only once. If you find yourself using it twice, it means you are doing something wrong.
* Recommended approach: Import the data and perform the train/test split like we always do. Then, perform the following for both train and test separately (pay attention to fit_transform and transform):
    * separate the text column and process it using the text-mining tutorials (pick any approach, though I would recommend the SCIKIT tutorial). Do not try to include text processing to the pipeline - it won't work.
    * separate the numeric and categorical columns, create pipelines for them like we always do.
    * concatenate the processed text and processed numerical/categorical columns using numpy's concatenate function. See the below example for the use of this function. 

# Section 1: (8 points in total)

## Data Prep (6 points)

In [1]:
import pandas as pd
import numpy as np

In [2]:
jobs = pd.read_csv('jobs_alldata.csv')

In [3]:
jobs.head(5)

,Salary,Job Description,Location,Min_years_exp,Technical,Comm,Travel
0,67206,Civil Service Title: Regional Director Mental ...,Remote,5,2,3,0
1,88313,The New York City Comptrollerâ€™s Office Burea...,Remote,5,2,4,10-15
2,81315,With minimal supervision from the Deputy Commi...,East campus,5,3,3,5-10
3,76426,OPEN TO CURRENT BUSINESS PROMOTION COORDINATOR...,East campus,1,1,3,0
4,55675,Only candidates who are permanent in the Princ...,Southeast campus,1,1,3,5-10


In [4]:
target = jobs['Salary']

In [5]:
jobs.isna().sum()

Salary             0
Job Description    0
Location           0
Min_years_exp      0
Technical          0
Comm               0
Travel             0
dtype: int64

In [6]:
from sklearn.model_selection import train_test_split
train_set, test_set = train_test_split(jobs, test_size=0.3)

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import FunctionTransformer

In [8]:
train_y = train_set[['Salary']]
test_y = test_set[['Salary']]

train_inputs = train_set.drop(['Salary'], axis=1)
test_inputs = test_set.drop(['Salary'], axis=1)


## Feature Engineering (1 points)

Create one NEW feature from existing data. You either transform a single variable, or create a new variable from existing ones. 

Grading: 
- 0.5 points for creating the new feature correctly
- 0.5 points for the justification of the new feature (i.e., why did you create this new feature)

In [9]:
#Create a flag for whether a location is remote or not

def new_rfcol(df):
    df1 = df.copy()
    df1['remote_flag'] = np.where(df1['Location'].str.lower() == 'remote', 1, 0)
    return df1[['remote_flag']]

In [10]:
train_set['Location'].value_counts()

HQ                  650
Remote              356
East campus         333
West campus         180
Southeast campus    170
Name: Location, dtype: int64

In [11]:
new_rfcol(train_set).value_counts()

remote_flag
0              1333
1               356
dtype: int64

In [12]:
train_inputs.dtypes

Job Description    object
Location           object
Min_years_exp       int64
Technical           int64
Comm                int64
Travel             object
dtype: object

In [13]:
# Identify the numerical columns
numeric_columns = train_inputs.select_dtypes(include=[np.number]).columns.to_list()
# Identify the categorical columns
categorical_columns = train_inputs.select_dtypes('object').columns.to_list()

In [14]:
categorical_columns.remove('Job Description')

In [15]:
categorical_columns

['Location', 'Travel']

In [16]:
numeric_columns

['Min_years_exp', 'Technical', 'Comm']

In [17]:
feat_eng_columns = ['Location']

In [18]:
train_inputs.dtypes

Job Description    object
Location           object
Min_years_exp       int64
Technical           int64
Comm                int64
Travel             object
dtype: object

In [19]:
numeric_transformer = Pipeline(steps=[
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])

In [20]:
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])

In [21]:
new_rfcol = Pipeline(steps=[('rfcol', FunctionTransformer(new_rfcol)),
                               ('scaler', StandardScaler())])

In [22]:
preprocessor = ColumnTransformer([
        ('num', numeric_transformer, numeric_columns),
        ('cat', categorical_transformer, categorical_columns),
        ('trans', new_rfcol, feat_eng_columns)],
        remainder='drop')

In [23]:
train_x = preprocessor.fit_transform(train_inputs)
train_x

array([[ 1.10216233,  0.63040116, -1.32675255, ...,  0.        ,
         0.        , -0.51678503],
       [ 1.10216233, -1.0152674 ,  0.97987128, ...,  0.        ,
         0.        ,  1.93504057],
       [-1.12125024, -1.0152674 ,  0.97987128, ...,  0.        ,
         0.        , -0.51678503],
       ...,
       [ 0.54630919, -1.0152674 , -1.32675255, ...,  0.        ,
         0.        ,  1.93504057],
       [-1.12125024, -1.0152674 , -0.17344063, ...,  0.        ,
         0.        , -0.51678503],
       [-1.12125024,  2.27606972,  0.97987128, ...,  1.        ,
         0.        , -0.51678503]])

In [24]:
train_x.shape, train_set.shape, train_y.shape

((1689, 13), (1689, 7), (1689, 1))

In [25]:
test_x = preprocessor.transform(test_inputs)
test_x

array([[-1.12125024, -0.19243312, -0.17344063, ...,  1.        ,
         0.        , -0.51678503],
       [ 1.10216233,  1.45323544, -1.32675255, ...,  0.        ,
         0.        ,  1.93504057],
       [-1.12125024, -0.19243312,  0.97987128, ...,  0.        ,
         1.        , -0.51678503],
       ...,
       [-0.00954396, -1.0152674 , -1.32675255, ...,  0.        ,
         0.        , -0.51678503],
       [ 1.10216233,  0.63040116, -1.32675255, ...,  0.        ,
         0.        , -0.51678503],
       [ 1.10216233, -1.0152674 ,  2.1331832 , ...,  0.        ,
         0.        , -0.51678503]])

In [26]:
test_x.shape, test_set.shape

((724, 13), (724, 7))

In [27]:
train_set

,Salary,Job Description,Location,Min_years_exp,Technical,Comm,Travel
2267,60230,The Division of Family and Child Health (DFCH)...,West campus,5,3,2,1-5
1448,119681,Only candidates who are permanent in the Admin...,Remote,5,1,4,1-5
778,61274,The Commission on Human Rights (the Commission...,East campus,1,1,4,0
660,47050,"â€¢\tUnder supervision, assist in the planting...",HQ,5,2,3,0
1827,52112,"Under general supervision, performs responsibl...",East campus,5,1,2,0
...,...,...,...,...,...,...,...
931,75463,The New York City Department of Environmental ...,HQ,1,2,3,0
804,54919,The Division of Economic and Financial Opportu...,HQ,5,2,4,0
2402,54294,The NYC Department of Environmental Protection...,Remote,4,1,2,0
202,148478,"The NYC Department of Design and Construction,...",West campus,1,1,3,1-5


In [28]:
test_set

,Salary,Job Description,Location,Min_years_exp,Technical,Comm,Travel
680,46607,"Under supervision, with latitude for independe...",East campus,1,2,3,10-15
2185,77722,Please read this posting carefully to make cer...,Remote,5,4,2,0
625,53215,Only candidates who are permanent in the Civil...,West campus,1,2,4,5-10
2024,74986,"DoITT provides for the sustained, efficient an...",HQ,1,2,5,0
1927,67283,The NYC Department of Environmental Protection...,West campus,5,1,4,0
...,...,...,...,...,...,...,...
222,116522,The NYC Department of Environmental Protection...,Southeast campus,5,2,4,0
1620,129270,The Comptroller's Bureau of Contract Administr...,East campus,1,2,4,0
1049,52000,***PLEASE NOTE APPLICANTS MUST BE PERMANENT IN...,West campus,3,1,2,0
1617,60496,The New York City Department of Environmental ...,East campus,5,3,2,0


In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vect = TfidfVectorizer(stop_words='english')
train_x_tr = tfidf_vect.fit_transform(train_set['Job Description'])

In [30]:
test_x_tr = tfidf_vect.transform(test_set['Job Description'])

In [31]:
train_x_tr.toarray()

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.05707497, 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.03534338, 0.        , ..., 0.        , 0.        ,
        0.        ]])

In [32]:
test_x_tr.toarray()

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.05341507, 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.03957879, 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]])

In [52]:
from sklearn.decomposition import TruncatedSVD
svd = TruncatedSVD(n_components=800, n_iter=5)

In [53]:
train_x_lsa = svd.fit_transform(train_x_tr)

In [54]:
train_x_lsa.shape

(1689, 800)

In [55]:
test_x_lsa = svd.transform(test_x_tr)

In [56]:
test_x_lsa.shape

(724, 800)

In [57]:
svd.explained_variance_ratio_.sum()

# Increasing components above 800 has a big impact on performance.  Keeping the 95% found.

0.9571020515804016

In [58]:
#Baseline
from sklearn.dummy import DummyRegressor
dummy_regr = DummyRegressor(strategy="mean")
dummy_regr.fit(train_x, train_y)

DummyRegressor()

In [59]:
from sklearn.metrics import mean_squared_error
dummy_train_pred = dummy_regr.predict(train_x)
baseline_train_mse = mean_squared_error(train_y, dummy_train_pred)
baseline_train_rmse = np.sqrt(baseline_train_mse)
print('Baseline Train RMSE: {}' .format(baseline_train_rmse))

Baseline Train RMSE: 29104.736404930834


In [60]:
dummy_test_pred = dummy_regr.predict(test_x)
baseline_test_mse = mean_squared_error (test_y, dummy_test_pred)
baseline_test_rmse = np.sqrt(baseline_test_mse)
print('Baseline Test RMSE: {}' .format(baseline_test_rmse))

Baseline Test RMSE: 29413.226614403862


In [61]:
# Merge the train dataset
train_combined = np.concatenate((train_x,train_x_lsa), axis=1)
train_combined

array([[ 1.10216233e+00,  6.30401159e-01, -1.32675255e+00, ...,
         2.15952969e-02, -1.19907183e-03, -4.95528019e-03],
       [ 1.10216233e+00, -1.01526740e+00,  9.79871283e-01, ...,
         5.69164302e-03,  7.30387558e-03, -2.47164671e-03],
       [-1.12125024e+00, -1.01526740e+00,  9.79871283e-01, ...,
         8.92552195e-03,  2.27451115e-02,  3.09031345e-02],
       ...,
       [ 5.46309188e-01, -1.01526740e+00, -1.32675255e+00, ...,
         7.65670896e-03,  2.50541920e-03,  3.92479794e-03],
       [-1.12125024e+00, -1.01526740e+00, -1.73440631e-01, ...,
        -3.09548883e-02, -2.86160201e-02, -4.42519408e-03],
       [-1.12125024e+00,  2.27606972e+00,  9.79871283e-01, ...,
        -2.83578152e-02,  2.82853268e-02,  1.83138020e-02]])

In [65]:
train_combined.shape
# shows the 1689 rows plus the 800 SVD columns and the 13 pipeline columns

(1689, 813)

In [66]:
# Merge the test dataset
test_combined = np.concatenate((test_x,test_x_lsa), axis=1)

In [67]:
test_combined.shape

(724, 813)

# Section 2: (7 points in total)

Build the following models:


## Decision Tree: (1 point)

In [70]:
#Test RMSE
test_pred = rnd_reg.predict(test_combined)
test_mse = mean_squared_error(test_y, test_pred)
test_rmse = np.sqrt(test_mse)
print('Test RMSE: {}' .format(test_rmse))

# Original: (n_estimators=300, max_depth=5,min_samples_leaf=5,n_jobs=-1)
# As I increased max_depth from 5 to 10, the overfitting worsened, but the RMSE decreased -> better model
# By then increasing min_samples_leaf = 10, the overfitting worsened, but the RMSE also increased
# Also, tried max_depth=15,min_samples_leaf=5 -> best model -- bad overfitting, but lowest RMSE of all attempts
# Started altering the n_estimators by 50, going down to 100.  300 -> 200 = lowest RMSE, trivial overfitting reduction
# This was the model with the lowest RMSE.  

Test RMSE: 19831.41538239328


## Voting regressor (2 points):

The voting regressor should have at least 3 individual models

In [72]:
#Train RMSE
train_pred = voting_reg.predict(train_combined)
train_mse = mean_squared_error(train_y, train_pred)
train_rmse = np.sqrt(train_mse)
print('Train RMSE: {}' .format(train_rmse))

Train RMSE: 12664.753956765775


In [73]:
#Test RMSE
test_pred = voting_reg.predict(test_combined)
test_mse = mean_squared_error(test_y, test_pred)
test_rmse = np.sqrt(test_mse)
print('Test RMSE: {}' .format(test_rmse))

# Original:
#dtree_reg = DecisionTreeRegressor(max_depth=20)
#svm_reg = SVR(kernel="rbf", C=10, epsilon=0.01, gamma='scale') 
#sgd_reg = SGDRegressor(max_iter=10000, tol=1e-3)
# Some overfitting, not as good as DT alone

# Increased C for SVR from 10 to 100 -> slightly worse RMSE, slight decrease in overfitting 
# Added penalty='elasticnet', l1_ratio=.25 to SGDRegressor -> worse RMSE -> slight decrease in overfitting
# Used the same param in DT -> much lower overfitting (3%), but worse RMSE (increase)
# Decreased max_iter from 10000 to 1000 for SGD -> Worse RMSE, lower overfitting

# Went back to original values 

Test RMSE: 19707.857117949443


## A Boosting model: (1 point)

Build either an Adaboost or a GradientBoost model

In [81]:
#Train RMSE
train_pred = gb_reg.predict(train_combined)
train_mse = mean_squared_error(train_y, train_pred)
train_rmse = np.sqrt(train_mse)
print('Train RMSE: {}' .format(train_rmse))

Train RMSE: 3209.780988049443


In [82]:
#Test RMSE
test_pred = gb_reg.predict(test_combined)
test_mse = mean_squared_error(test_y, test_pred)
test_rmse = np.sqrt(test_mse)
print('Test RMSE: {}' .format(test_rmse))

# Original: GradientBoostingRegressor(n_estimators=40, learning_rate=0.2) -- OVerfitting, worse than DT model RMSE (higher)
# Increased estimators from 40 to 100 and learning rate from 0.2 to 0.3 -> Overfitting, still there, much better RMSE
# Increased estimators from 100 to 120 -> Better train, worse test 
# Decreased estimated to 90 , higher RMSE (worse)

# Best model seems to be 100 estimators and a learning rate of .4



Test RMSE: 17387.68941060901


## Neural network: (1 point)

In [102]:
#Train RMSE
train_pred = dnn_reg.predict(train_combined)
train_mse = mean_squared_error(train_y, train_pred)
train_rmse = np.sqrt(train_mse)
print('Train RMSE: {}' .format(train_rmse))

Train RMSE: 11400.26789608273


In [103]:
#Test RMSE
test_pred = dnn_reg.predict(test_combined)
test_mse = mean_squared_error(test_y, test_pred)
test_rmse = np.sqrt(test_mse)
print('Test RMSE: {}' .format(test_rmse))

# Original: MLPRegressor(hidden_layer_sizes=(10,5), max_iter=500) /No overfitting  Terrible RMSE (45667)
# Going down to one layer made performance much worse (82388)
# Went to 2 layers of 10 neurons: improved MRSE (28350) 
# Went to 2 layers of 20 neurons: improved MRSE (22994)
# Went to 3 layers of 30/20/10 neurons: improved MRSE (18585)
# 4 layers - pipe shape (all 10 neurons) No overfitting, best RMSE - Adding layers or neurons did not change much

Test RMSE: 18941.16529737021


## Grid search (2 points)

Perform either a full or randomized grid search on any model you want. There has to be at least two parameters for the search. 

In [104]:
from sklearn.model_selection import RandomizedSearchCV
param_grid = [
    {'min_samples_leaf': np.arange(5, 20), 
     'max_depth': np.arange(5,25),
     'splitter': ['random', 'best']
    }
  ]
tree_reg = DecisionTreeRegressor()
grid_search = RandomizedSearchCV(tree_reg, param_grid, cv=6, n_iter=15,
                           scoring='neg_mean_squared_error', verbose=1,
                           return_train_score=True)
grid_search.fit(train_combined, train_y)

Fitting 6 folds for each of 15 candidates, totalling 90 fits


RandomizedSearchCV(cv=6, estimator=DecisionTreeRegressor(), n_iter=15,
                   param_distributions=[{'max_depth': array([ 5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21,
       22, 23, 24]),
                                         'min_samples_leaf': array([ 5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]),
                                         'splitter': ['random', 'best']}],
                   return_train_score=True, scoring='neg_mean_squared_error',
                   verbose=1)

In [105]:
cvres = grid_search.cv_results_
for mean_score, params in zip(cvres["mean_test_score"], cvres["params"]):
    print(np.sqrt(-mean_score), params)

27288.730054463373 {'splitter': 'best', 'min_samples_leaf': 13, 'max_depth': 23}
26645.84100107623 {'splitter': 'random', 'min_samples_leaf': 12, 'max_depth': 15}
27495.446591180105 {'splitter': 'random', 'min_samples_leaf': 15, 'max_depth': 18}
27770.319540349992 {'splitter': 'best', 'min_samples_leaf': 19, 'max_depth': 15}
27717.778624575443 {'splitter': 'random', 'min_samples_leaf': 6, 'max_depth': 24}
27126.556458210904 {'splitter': 'random', 'min_samples_leaf': 5, 'max_depth': 12}
27516.46501311653 {'splitter': 'random', 'min_samples_leaf': 16, 'max_depth': 20}
27046.756583948794 {'splitter': 'best', 'min_samples_leaf': 15, 'max_depth': 13}
27202.42775412081 {'splitter': 'random', 'min_samples_leaf': 19, 'max_depth': 9}
27836.779253248216 {'splitter': 'random', 'min_samples_leaf': 5, 'max_depth': 14}
27713.37476481464 {'splitter': 'random', 'min_samples_leaf': 16, 'max_depth': 23}
27756.882783922163 {'splitter': 'random', 'min_samples_leaf': 19, 'max_depth': 8}
27622.124013330682 

In [106]:
grid_search.best_params_

{'splitter': 'random', 'min_samples_leaf': 12, 'max_depth': 15}

In [107]:
grid_search.best_estimator_

DecisionTreeRegressor(max_depth=15, min_samples_leaf=12, splitter='random')

In [108]:
#Train RMSE
train_pred = grid_search.best_estimator_.predict(train_combined)
train_mse = mean_squared_error(train_y, train_pred)
train_rmse = np.sqrt(train_mse)
print('Train RMSE: {}' .format(train_rmse))

Train RMSE: 18623.561489790576


In [109]:
#Test RMSE
test_pred = grid_search.best_estimator_.predict(test_combined)
test_mse = mean_squared_error(test_y, test_pred)
test_rmse = np.sqrt(test_mse)
print('Test RMSE: {}' .format(test_rmse))

# Original: param_grid = [{'min_samples_leaf': np.arange(1, 10), 'max_depth': np.arange(25,125),'splitter': ['random', 'best']}
# Bad overfitting (6000 to 22000) and worse RMSE compared to other models
# Updated to: {'min_samples_leaf': np.arange(1, 15), 'max_depth': np.arange(5,75)}  -> Worse overfitting, slightly better RMSE
# Updated to: {'min_samples_leaf': np.arange(10, 20), 'max_depth': np.arange(5,25) -> Much less overfitting, worse RMSE
# Updated to: 'min_samples_leaf': np.arange(5, 20) -> Best result for this Grid Search

Test RMSE: 26011.418858830228


# Discussion (5 points in total)


## List the train and test values of each model you built (2 points)

## Which model performs the best and why? (0.5 points) 
## How does it compare to baseline? (0.5 points)

Hint: The best model is the one that has the lowest TEST RMSE value (regardless of any of the training values). If you select your model based on TRAIN values, you will lose points.

## Is there any evidence of overfitting in the best model, why or why not? If there is, what did you do about it? (1 point)

## Is there any overfitting in the other models (besides the best model), why or why not? If there is, what did you do about it? (1 point)